In [6]:
import rasterio as rio
from pathlib import Path 
import os
import numpy as np

Let's first read the imgs

In [7]:
imgs = list(Path('/Data_large/marine/Datasets/VENuS/ds_L0/perfect').glob('*.tif'))
print('Listing the number of imgs:')
print(len(imgs))

Listing the number of imgs:
282


Definition of utils functions:

In [8]:
def read_tif(file_path, band_indices):
    """
    Reads specified bands from a TIFF file.

    Parameters:
    - file_path (str): Path to the .tif file.
    - band_indices (list of int): Indices of the bands to read.

    Returns:
    - dict: A dictionary where keys are band indices and values are the corresponding band data.
    """
    data = {}
    with rio.open(file_path) as src:
        crs = src.crs
        transform = src.transform
        
        for index in band_indices:
            data[index] = src.read(index)
    return data, crs, transform


def save_tif(data, output_dir, file_name, crs, transform):
    """
    Saves data to a TIFF file in the specified directory, preserving the CRS and transform.
    This version adjusts for a single-band array to include an explicit band dimension correctly.
    
    Parameters:
    - data (dict): Data to save, where keys are band numbers and values are band data arrays.
    - output_dir (str): Directory to save the file.
    - file_name (str): Name for the output file.
    - crs: Coordinate reference system to be used for the output file.
    - transform: Affine transform to be used for the output file.
    """
    if not data:
        raise ValueError("No data to save.")
    
    # Determine the number of bands and set data type
    num_bands = len(data)
    first_band = next(iter(data.values()))
    dtype = first_band.dtype
    
    # Update meta based on number of bands
    meta = {
        'driver': 'GTiff',
        'height': first_band.shape[0],
        'width': first_band.shape[1],
        'count': num_bands,
        'dtype': dtype,
        'crs': crs,
        'transform': transform
    }
    
    # Open the output file with the correct metadata
    with rio.open(f'{output_dir}/{file_name}', 'w', **meta) as dst:
        for i, band_data in enumerate(data.values(), start=1):
            # Check if it's a single-band and 2D array
            if band_data.ndim == 2 and num_bands == 1:
                # Write the band data while specifying band index
                dst.write(band_data, 1)
            else:
                # Write the band data normally
                dst.write(band_data, i)

### Export Singularly the bands of VENUS

In [4]:
Savefolder = '/Data_large/marine/Datasets/VENuS/ds_L0'

bands_selection = [1,2,3,4,5,6,7,8,9,10,11,12]

for band in bands_selection:
    for im in imgs:
        # Reading data
        data, crs, transform = read_tif(im, band_indices=[band])
        fname = im.name
        # Saving data
        outdir = f'{Savefolder}/perfect_b{band}'
        os.makedirs(outdir, exist_ok=True)
        save_tif(data, output_dir=outdir, file_name=fname, crs=crs, transform=transform)

## Export Multiple Bands of VENUS:

In [9]:

Savefolder = '/Data_large/marine/Datasets/VENuS/ds_L0'

"""
bands_selection = [1,2,3,4,5,6,7,8,9,10,11,12]
# Double bands:
bands_selection = [5,10]
bands_selection = [5,12]
# Triple bands:
bands_selection = [3,4,7]
bands_selection = [5,10,11]
# Four bands:
bands_selection = [3,4,7,11]
bands_selection = [3,5,7,11]
"""

BANDS_SELECTIONS = [[5,10],
                    [5,12],
                    [3,4,7],
                    [5,10,11],
                    [3,4,7,11],
                    [3,5,7,11]]

BANDS_SELECTIONS = [[3,8,11],
                    [3,10,11]]

for bands_selection in BANDS_SELECTIONS:
    stringa_bands = '_'.join([str(i) for i in bands_selection])
    for im in imgs:
        # Reading data
        data, crs, transform = read_tif(im, band_indices=bands_selection)
        fname = im.name
        # Saving data
        outdir = f'{Savefolder}/perfect_b{stringa_bands}'
        os.makedirs(outdir, exist_ok=True)
        save_tif(data, output_dir=outdir, file_name=fname, crs=crs, transform=transform)